Fabric notebook: Load Reference Data into Lakehouse Tables

This notebook reads JSONL files from the lh_insurance Lakehouse
Files/reference_data/ folder and creates Delta tables with proper schemas.

Prerequisites:
  - Attach this notebook to the "lh_insurance" Lakehouse
  - Upload the reference_data/*.jsonl files to Files/reference_data/

To run: Execute all cells in order.

## Configuration

In [ ]:
LAKEHOUSE_NAME       = "lh_insurance"
REFERENCE_DATA_PATH  = "Files/reference_data"

## Schema Definitions

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType, DoubleType,
    BooleanType, DateType, TimestampType, ArrayType, LongType
)

SCHEMAS = {
    "offices": StructType([
        StructField("office_id", StringType(), False),
        StructField("name", StringType(), False),
        StructField("office_type", StringType(), False),
        StructField("address", StringType(), False),
        StructField("city", StringType(), False),
        StructField("state", StringType(), False),
        StructField("zip_code", StringType(), False),
        StructField("latitude", DoubleType(), False),
        StructField("longitude", DoubleType(), False),
        StructField("timezone", StringType(), False),
        StructField("phone", StringType(), True),
        StructField("adjuster_capacity", IntegerType(), True),
        StructField("has_siu_unit", BooleanType(), True),
    ]),

    "policyholders": StructType([
        StructField("policyholder_id", StringType(), False),
        StructField("policyholder_number", StringType(), False),
        StructField("full_name", StringType(), False),
        StructField("policyholder_type", StringType(), False),
        StructField("email", StringType(), True),
        StructField("phone", StringType(), True),
        StructField("address", StringType(), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("zip_code", StringType(), True),
        StructField("date_of_birth", StringType(), True),
        StructField("risk_score", IntegerType(), True),
    ]),

    "insured_assets": StructType([
        StructField("asset_id", StringType(), False),
        StructField("asset_number", StringType(), False),
        StructField("asset_type", StringType(), False),
        StructField("asset_description", StringType(), False),
        StructField("make", StringType(), True),
        StructField("model", StringType(), True),
        StructField("year", IntegerType(), False),
        StructField("vin", StringType(), True),
        StructField("address", StringType(), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("estimated_value", DoubleType(), True),
        StructField("condition", StringType(), True),
        StructField("last_inspection_date", StringType(), True),
    ]),

    "adjusters": StructType([
        StructField("adjuster_id", StringType(), False),
        StructField("employee_id", StringType(), False),
        StructField("first_name", StringType(), False),
        StructField("last_name", StringType(), False),
        StructField("email", StringType(), True),
        StructField("phone", StringType(), True),
        StructField("license_number", StringType(), False),
        StructField("license_state", StringType(), False),
        StructField("specializations", ArrayType(StringType()), True),
        StructField("hire_date", StringType(), True),
        StructField("status", StringType(), False),
        StructField("home_office_id", StringType(), False),
        StructField("supervisor_email", StringType(), True),
        StructField("max_active_claims", IntegerType(), True),
    ]),

    "policies": StructType([
        StructField("policy_id", StringType(), False),
        StructField("policy_number", StringType(), False),
        StructField("policyholder_id", StringType(), False),
        StructField("asset_id", StringType(), False),
        StructField("policy_type", StringType(), False),
        StructField("coverage_amount", DoubleType(), True),
        StructField("deductible", DoubleType(), True),
        StructField("premium_annual", DoubleType(), True),
        StructField("effective_date", StringType(), True),
        StructField("expiration_date", StringType(), True),
        StructField("status", StringType(), False),
        StructField("underwriter", StringType(), True),
    ]),

    "claims": StructType([
        StructField("claim_id", StringType(), False),
        StructField("claim_number", StringType(), False),
        StructField("policy_id", StringType(), False),
        StructField("policyholder_id", StringType(), False),
        StructField("asset_id", StringType(), False),
        StructField("adjuster_id", StringType(), False),
        StructField("office_id", StringType(), False),
        StructField("claim_type", StringType(), False),
        StructField("description", StringType(), True),
        StructField("incident_date", StringType(), True),
        StructField("filed_date", StringType(), True),
        StructField("status", StringType(), False),
        StructField("estimated_loss", DoubleType(), True),
        StructField("approved_amount", DoubleType(), True),
        StructField("priority", StringType(), True),
        StructField("incident_latitude", DoubleType(), True),
        StructField("incident_longitude", DoubleType(), True),
    ]),

    "claim_events": StructType([
        StructField("claim_event_id", StringType(), False),
        StructField("event_number", StringType(), False),
        StructField("claim_id", StringType(), False),
        StructField("adjuster_id", StringType(), False),
        StructField("event_type", StringType(), False),
        StructField("description", StringType(), True),
        StructField("status", StringType(), False),
        StructField("created_at", StringType(), True),
        StructField("completed_at", StringType(), True),
        StructField("notes", StringType(), True),
        StructField("cost_usd", DoubleType(), True),
    ]),

    "asset_inspections": StructType([
        StructField("inspection_id", StringType(), False),
        StructField("asset_id", StringType(), False),
        StructField("office_id", StringType(), False),
        StructField("inspection_type", StringType(), False),
        StructField("status", StringType(), False),
        StructField("scheduled_date", StringType(), True),
        StructField("completed_date", StringType(), True),
        StructField("appraised_value", DoubleType(), True),
        StructField("inspector_notes", StringType(), True),
        StructField("condition_rating", StringType(), True),
    ]),
}

# Primary key column for each table
PRIMARY_KEYS = {
    "offices": "office_id",
    "policyholders": "policyholder_id",
    "insured_assets": "asset_id",
    "adjusters": "adjuster_id",
    "policies": "policy_id",
    "claims": "claim_id",
    "claim_events": "claim_event_id",
    "asset_inspections": "inspection_id",
}

## Load JSONL Files into Lakehouse Tables

In [ ]:
# Order matters: load tables with no FK dependencies first
LOAD_ORDER = [
    "offices",
    "policyholders",
    "insured_assets",
    "adjusters",
    "policies",
    "claims",
    "claim_events",
    "asset_inspections",
]

results = []

for table_name in LOAD_ORDER:
    file_path = f"{REFERENCE_DATA_PATH}/{table_name}.jsonl"
    schema = SCHEMAS[table_name]
    pk = PRIMARY_KEYS[table_name]

    try:
        # Read JSONL (JSON Lines format = one JSON object per line)
        df = spark.read.schema(schema).json(file_path)

        # Verify no null primary keys
        null_pk_count = df.filter(df[pk].isNull()).count()
        if null_pk_count > 0:
            print(f"⚠ WARNING: {table_name} has {null_pk_count} null primary keys ({pk})")

        # Verify primary key uniqueness
        total = df.count()
        distinct_pk = df.select(pk).distinct().count()
        if total != distinct_pk:
            print(f"⚠ WARNING: {table_name} has {total - distinct_pk} duplicate primary keys ({pk})")

        # Write as Delta table (overwrite to support re-runs)
        df.write.format("delta").mode("overwrite").saveAsTable(table_name)

        results.append({"table": table_name, "rows": total, "pk": pk, "status": "✓ loaded"})
        print(f"  ✓ {table_name}: {total} rows loaded (PK: {pk})")

    except Exception as e:
        results.append({"table": table_name, "rows": 0, "pk": pk, "status": f"✗ {str(e)[:80]}"})
        print(f"  ✗ {table_name}: FAILED — {e}")

## Summary

In [ ]:
# from pyspark.sql import Row

# summary_df = spark.createDataFrame([Row(**r) for r in results])
# display(summary_df)

## Verification: Row Counts & Sample Data

In [ ]:
# print("=" * 60)
# print("TABLE ROW COUNTS")
# print("=" * 60)
# for table_name in LOAD_ORDER:
#     try:
#         count = spark.table(table_name).count()
#         print(f"  {table_name:<25} {count:>6} rows")
#     except Exception as e:
#         print(f"  {table_name:<25} ERROR: {e}")

In [ ]:
# # Display sample records from key tables
# for table_name in ["offices", "policyholders", "claims", "policies"]:
#     print(f"\n{'=' * 60}")
#     print(f"SAMPLE: {table_name} (first 3 rows)")
#     print("=" * 60)
#     display(spark.table(table_name).limit(3))